# Lab 06 - Develop with Snowpark, then deploy
Copyright © 2026 Innovation In Software Corporation. All rights reserved.

Run **Part A only** of `06-SnowparkAndDeployment.sql` first. Import this notebook into Snowflake as ADV_LAB, choose ADV_WH as query warehouse and the warehouse Python runtime available in your account. Add `snowflake-snowpark-python` in the notebook Packages settings without a version pin. Every student uses the same setup. No local Python or credentials are required.

Instructor executes each cell; students repeat and inspect its result. The notebook uses **Snowpark**, not PySpark. SQL already solves the orders problem; the purpose is learning the DataFrame development and deployment workflow.

Existing Bronze input and Silver targets are our contract. Development cells read the committed inbox from Labs 03-04 and build plans. They do not consume the stream. One marked trial transaction writes the Snowpark scratch outputs, merges, then always rolls back. Do not schedule this development notebook: it deliberately contains displays and a rollback.

In [ ]:
USE ROLE ADV_LAB;

In [ ]:
from snowflake.snowpark.context import get_active_session
import snowflake.snowpark
session = get_active_session()


In [ ]:
role = session.sql("SELECT CURRENT_ROLE()").collect()[0][0]
assert role == "ADV_LAB", "Use the student ADV_LAB role"
session.use_database("ADV_ORDERS")
session.use_schema("SILVER")
session.use_warehouse("ADV_WH")


In [ ]:
# Common session zone for history/LTZ display; fixture event times already include Z.

session.sql("ALTER SESSION SET TIMEZONE = 'UTC'").collect()
# Separate notebook session: set its own tag for the SQL retrieval example.

session.sql("ALTER SESSION SET QUERY_TAG = 'ADV_2026_DAY1'").collect()
print("Notebook Snowpark version:", snowflake.snowpark.__version__)


## 1. Read existing deliveries
`session.table` creates a DataFrame plan. `.show()` executes the plan for inspection; `.count()` executes an aggregate. Assignment alone does not load all events into Python memory. Expect inbox=10 on the first pass, or 11 after completing the replay. Current orders=3 and the stream must be empty before this development exercise.

In [ ]:
"""Reusable Snowpark transformations for the Day 1 order-event pipeline.
Copyright © 2026 Innovation In Software Corporation. All rights reserved.
"""
from snowflake.snowpark.functions import (
    col, sql_expr, row_number, when_matched, when_not_matched
)
from snowflake.snowpark.window import Window

ORDER_COLUMNS = ["ORDER_ID", "EVENT_ID", "VERSION", "EVENT_TIME", "STATUS", "AMOUNT", "REGION"]
EVENT_COLUMNS = ["EVENT_ID", "ORDER_ID", "VERSION", "EVENT_TIME", "STATUS", "AMOUNT",
                 "REGION", "REGION_LABEL", "ACTIVE_AMOUNT_USD", "REJECT_REASON",
                 "SOURCE_FILE", "SOURCE_ROW"]

raw = session.table("SILVER.EVENT_INBOX")
assert raw.count() in (10, 11), "Complete Labs 01-05 before this notebook"
assert session.table("BRONZE.EVENTS_STREAM").count() == 0, "Finish processing pending events first"
raw.select("PAYLOAD", "SOURCE_FILE", "SOURCE_ROW").show()


## 2. Convert to typed columns
Define one reusable transformation, call it, then inspect its schema and rows. e3 amount becomes NULL. For this classroom example, VERSION uses NUMBER(3,1). Valid revisions are whole numbers; 1.5 still fails validation. This limited precision is a teaching simplification, not a general validator for arbitrary source versions. `select_expr` uses SQL expressions inside a Snowpark DataFrame, without collecting the event data.

In [ ]:
def type_events(df):
    # select_expr builds a DataFrame plan; it does not collect the event rows.
    # Classroom versions use NUMBER(3,1) to keep the displayed values compact.

    return df.select_expr(
        "NULLIF(TRIM(payload:event_id::STRING), '') AS EVENT_ID",
        "NULLIF(TRIM(payload:order_id::STRING), '') AS ORDER_ID",
        "TRY_TO_DECIMAL(payload:version::STRING, 3, 1) AS VERSION",
        "TRY_TO_TIMESTAMP_TZ(payload:event_time::STRING) AS EVENT_TIME",
        "payload:status::STRING AS STATUS",
        "TRY_TO_DECIMAL(payload:amount::STRING, 12, 2) AS AMOUNT",
        "payload:region::STRING AS REGION",
        "SOURCE_FILE", "SOURCE_ROW")

typed = type_events(raw)
print(typed.schema)
typed.select("EVENT_ID", "ORDER_ID", "VERSION", "AMOUNT").sort("EVENT_ID").show()


## 3. Apply the business validation rules
`with_column` adds a classification expression to the plan. The CASE rules match Lab 03. Expect exactly e3 / BAD_AMOUNT; bad data stays in the inbox.

In [ ]:
def classify_events(df):
    # Reuse the business rules developed in SQL; express a new DataFrame column.

    return df.with_column("REJECT_REASON",
                          sql_expr("""CASE
                                        WHEN event_id IS NULL OR order_id IS NULL THEN 'MISSING_KEY'
                                        WHEN version IS NULL OR version < 1 OR version <> TRUNC(version) THEN 'BAD_VERSION'
                                        WHEN event_time IS NULL THEN 'BAD_TIME'
                                        WHEN status IS NULL OR status NOT IN ('PLACED', 'PAID', 'CANCELLED') THEN 'BAD_STATUS'
                                        WHEN amount IS NULL OR amount < 0 THEN 'BAD_AMOUNT'
                                        WHEN region IS NULL OR region NOT IN ('US', 'EU') THEN 'BAD_REGION'
                                        ELSE NULL
                                    END"""))

classified = classify_events(typed)
rejected = classified.filter(col("REJECT_REASON").is_not_null())
rejected.select("EVENT_ID", "REJECT_REASON").show()
assert rejected.count() == 1


## 4. Enrich with a reference join
Join the region labels and derive active amount. Cancellation has active amount zero. This is a left join so missing reference data would not silently remove an event. Inspect both the reference and the enriched result.

In [ ]:
def enrich_events(df_classified, df_reference):
    enriched = df_classified.join(
        df_reference, df_classified["REGION"] == df_reference["REGION_CODE"], "left")
    return enriched.with_column("ACTIVE_AMOUNT_USD", sql_expr("IFF(status = 'CANCELLED', 0, amount)"))

reference = session.table("SILVER.REGION_REFERENCE")
reference.show()

enriched = enrich_events(classified, reference)
enriched.select("EVENT_ID", "REGION", "REGION_LABEL", "STATUS", "ACTIVE_AMOUNT_USD").sort("EVENT_ID").show()


## 5. Check conflicts and select one latest state per order
Stop if one immutable event ID has conflicting valid payloads. The existing SQL conflict view has the same classroom limitation noted in Lab 03: TO_JSON is not a canonical JSON comparison.

The first window selects a representative delivery per event ID, keeping delivery lineage for inspection. The second window selects the latest business state per order. Neither window deletes inbox rows. Expect o1/e4/120, o2/e5/CANCELLED/50 and o3/e8/45.

In [ ]:
def select_candidates(df):
    good = df.filter(col("REJECT_REASON").is_null())

    # Pick a representative delivery of each immutable event; keep inbox history.
    # File/row identifies delivery lineage, not business-state precedence.

    unique = good.with_column("RN", row_number().over(Window.partition_by("EVENT_ID").order_by(col("SOURCE_FILE"), col("SOURCE_ROW"))))
    unique = unique.filter(col("RN") == 1).drop("RN")

    # Now choose the latest business state using the same SQL precedence rule.

    latest = unique.with_column("RN", row_number().over(Window.partition_by("ORDER_ID").order_by(col("VERSION").desc(), col("EVENT_TIME").desc(), col("EVENT_ID").desc())))
    return latest.filter(col("RN") == 1).select(*ORDER_COLUMNS)

assert session.table("SILVER.EVENT_CONFLICTS").count() == 0, "Resolve event-ID conflicts before continuing"

candidates = select_candidates(enriched)
candidates.sort("ORDER_ID").show()

assert candidates.count() == 3


## 6. Compare with the established SQL result
Show the matching rows through DataFrame `intersect`. Also check total counts and both differences: matches alone would hide extra rows. This develops transformations without changing the accepted current state.

In [ ]:
baseline = session.table("SILVER.ORDERS_CURRENT").select(*ORDER_COLUMNS)

candidates.intersect(baseline).sort("ORDER_ID").show()

assert baseline.count() == 3

assert candidates.subtract(baseline).count() == 0

assert baseline.subtract(candidates).count() == 0


## 7. Express MERGE through Snowpark
`Table.merge` uses `when_matched(condition)` and `when_not_matched()` clauses. Matching identifies the order; the additional condition tests whether its state is newer. Define the function first. Definition alone does not execute MERGE.

In [ ]:
def merge_current(sf_session, df_candidates):
    target = sf_session.table("SILVER.ORDERS_CURRENT")
    newer = (
        (df_candidates["VERSION"] > target["VERSION"])
        | ((df_candidates["VERSION"] == target["VERSION"])
           & (df_candidates["EVENT_TIME"] > target["EVENT_TIME"]))
        | ((df_candidates["VERSION"] == target["VERSION"])
           & (df_candidates["EVENT_TIME"] == target["EVENT_TIME"])
           & (df_candidates["EVENT_ID"] > target["EVENT_ID"]))
    )
    updates = {name: df_candidates[name] for name in ORDER_COLUMNS if name != "ORDER_ID"}
    inserts = {name: df_candidates[name] for name in ORDER_COLUMNS}

    # Table.merge is an action: Snowflake executes MERGE and returns row counts.

    return target.merge(df_candidates, target["ORDER_ID"] == df_candidates["ORDER_ID"], [
        when_matched(newer).update(updates),
        when_not_matched().insert(inserts)
    ])


## 8. Try the writes as one reversible unit
Both scratch destinations were created by Part A of the SQL script. Record their baseline counts. Inside the trial, replace their derived rows using DML, then call the merge function. The current states already agree, so expect zero inserted and updated orders. `finally` always rolls back, even if an inspection fails. Run this complete cell together; never insert DDL into this transaction.

In [ ]:
before_events = session.table("SILVER.SNOWPARK_EVENTS").count()
before_candidates = session.table("SILVER.SNOWPARK_CANDIDATES").count()

session.sql("BEGIN TRANSACTION").collect()

try:
    session.table("SILVER.SNOWPARK_EVENTS").delete()
    enriched.select(*EVENT_COLUMNS).write.insert_into("SILVER.SNOWPARK_EVENTS")

    session.table("SILVER.SNOWPARK_CANDIDATES").delete()
    candidates.write.insert_into("SILVER.SNOWPARK_CANDIDATES")

    result = merge_current(session, session.table("SILVER.SNOWPARK_CANDIDATES"))
    print(result)

    session.table("SILVER.SNOWPARK_CANDIDATES").sort("ORDER_ID").show()
finally:
    session.sql("ROLLBACK").collect()

assert session.table("SILVER.SNOWPARK_EVENTS").count() == before_events
assert session.table("SILVER.SNOWPARK_CANDIDATES").count() == before_candidates
assert session.table("BRONZE.EVENTS_STREAM").count() == 0


## 9. Assemble the verified functions into a handler
Now add transaction control, one stream-to-inbox INSERT, and the conflict guard. The transformation functions are the ones already developed above. Unlike the trial, a successful handler commits; an exception rolls back.

Define `main` below but do not call it in this development notebook. `Snowpark-OrderTransform.py` is the complete reusable source. Part B of the SQL script deploys identical source inside a Python 3.12 procedure with an unpinned Snowpark dependency, calls it, then switches the existing task graph. SQL is retained for transaction control and stream capture; transformation, scratch-table DML and MERGE use Snowpark APIs.

In [ ]:
def main(sf_session):
    # Only SQL transaction control and stream capture remain sf_session.sql calls.
    # All destinations already exist; no DDL belongs inside this transaction.

    sf_session.sql("BEGIN TRANSACTION").collect()
    try:
        # Consume this stream into the inbox ONCE per transaction.

        sf_session.sql("""INSERT INTO SILVER.EVENT_INBOX
            SELECT payload, source_file, source_row, scan_time
            FROM BRONZE.EVENTS_STREAM""").collect()
        if sf_session.table("SILVER.EVENT_CONFLICTS").count():
            raise ValueError("Conflicting payloads for immutable event_id")

        typed = type_events(sf_session.table("SILVER.EVENT_INBOX"))
        classified = classify_events(typed)
        enriched = enrich_events(classified, sf_session.table("SILVER.REGION_REFERENCE"))
        candidates = select_candidates(enriched)

        sf_session.table("SILVER.SNOWPARK_EVENTS").delete()
        enriched.select(*EVENT_COLUMNS).write.insert_into("SILVER.SNOWPARK_EVENTS")
        sf_session.table("SILVER.SNOWPARK_CANDIDATES").delete()
        candidates.write.insert_into("SILVER.SNOWPARK_CANDIDATES")
        result = merge_current(sf_session, sf_session.table("SILVER.SNOWPARK_CANDIDATES"))
        sf_session.sql("COMMIT").collect()
        return f"Applied with Snowpark: {result}"
    except Exception:
        sf_session.sql("ROLLBACK").collect()
        raise


## 10. Return to SQL deployment
Run Parts B and C of `06-SnowparkAndDeployment.sql` in order, pausing at asynchronous history checks. Deployment resolves the latest available Snowpark package once when the permanent procedure is created; calls do not upgrade it automatically. Verify the resolved dependency and business results before teaching with a new version.

Optional comparison: `Snowpark-Pipeline.ipynb` contains the same reusable functions with a single processing call, without development rollback/displays. `Optional-NotebookPipeline.sql` describes scheduling that execution notebook. Use either the procedure task or the notebook task as consumer, never both concurrently. Hosted notebook import and headless execution require a separate account rehearsal; they are not certified by the handler tests.